# 01_build_feature_catalog_and_area_inventory

**Role.** Feature registry, mapped-area inventory, and selected feature-set validation.

**Pipeline version.** Reproducible scientific pipeline v2 for Dak Lak 2024 coffee mapping Paper 1.


In [ ]:
# =============================================================================
# REPRODUCIBILITY BOOTSTRAP: Coffee Paper 1 pipeline v2
# =============================================================================
from pathlib import Path
import os, sys, json, warnings
import numpy as np

# Locate project root robustly whether the notebook is opened from project root
# or from the notebooks/ folder.
_candidate_roots = [Path.cwd().resolve()] + list(Path.cwd().resolve().parents)
PROJECT_ROOT = next((p for p in _candidate_roots if (p / "config" / "paper1_config.yaml").exists()), Path.cwd().resolve())
os.chdir(PROJECT_ROOT)
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from coffeemap.config import load_config, ensure_project_dirs, class_info, class_colors, coffee_class_ids
from coffeemap.manifest import init_run_manifest, append_manifest_note
from coffeemap.plotting import set_publication_style

CONFIG = load_config(PROJECT_ROOT / "config" / "paper1_config.yaml")
PATHS = ensure_project_dirs(CONFIG, PROJECT_ROOT)
CLASS_INFO = class_info(CONFIG)
CLASS_COLORS = class_colors(CONFIG)
CLASS_IDS = sorted(CLASS_INFO.keys())
CLASS_NAMES = [CLASS_INFO[i] for i in CLASS_IDS]
COFFEE_CLASSES = coffee_class_ids(CONFIG)
RANDOM_SEED = int(CONFIG.get("project", {}).get("random_seed", 42))
np.random.seed(RANDOM_SEED)

TABLES_DIR = PATHS["tables_dir"]
FIGURES_DIR = PATHS["figures_dir"]
SUPPLEMENTARY_DIR = PATHS["supplementary_dir"]
METADATA_DIR = PATHS["metadata_dir"]
INPUT_DIR = PATHS["input_dir"]

NOTEBOOK_NAME = "01_build_feature_catalog_and_area_inventory.ipynb"
MANIFEST = init_run_manifest(CONFIG, PROJECT_ROOT, notebook_name=NOTEBOOK_NAME)
set_publication_style(font="Arial", dpi=600)

print(f"Project root: {PROJECT_ROOT}")
print(f"Notebook: {NOTEBOOK_NAME}")
print(f"Classes: {len(CLASS_IDS)} | Coffee classes: {COFFEE_CLASSES} | Random seed: {RANDOM_SEED}")


## Reproducibility contract

This notebook follows the project-level configuration in `config/paper1_config.yaml` and writes outputs only under `results/`.

Key safeguards used in this pipeline:

- class IDs, class names, colors, paths, random seed, and coffee class definitions come from one config file;
- each notebook refreshes `results/metadata/run_manifest.json`;
- feature selection must use training data only;
- validation data are reserved for final assessment;
- Olofsson-style estimates are reported as **area-weighted error-adjusted estimates** unless a mapped-class stratified area-assessment sample is available;
- RF uncertainty is interpreted as **RF vote-based class probability**, not calibrated posterior probability.


In [2]:
# =============================================================================
# Pipeline-level imports commonly used by downstream cells
# =============================================================================
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

from coffeemap.io import find_file, read_table, write_table
from coffeemap.schema import (
    detect_column, detect_label_columns, assert_class_ids,
    class_count_table, warn_if_balanced, extract_probability_columns,
    assert_probability_matrix,
)
from coffeemap.metrics import classification_summary, overall_metrics, shannon_entropy, probability_margin
from coffeemap.validation import audit_validation_predictions
from coffeemap.olofsson import error_matrix_counts, area_adjustment, binary_coffee_area_adjustment

SEARCH_DIRS = [INPUT_DIR, PATHS["interim_dir"], TABLES_DIR, SUPPLEMENTARY_DIR, PROJECT_ROOT]
print("Reproducible pipeline helpers loaded.")


Reproducible pipeline helpers loaded.


In [ ]:
# =============================================================================
# TABLE 5: mapped area by land-cover class and coffee production system
# =============================================================================
class_area_path = find_file("Table_AreaStatistics_DakLak2024.csv", SEARCH_DIRS, required=True)
class_area = read_table(class_area_path)

required = {"class_id", "area_ha"}
missing = required - set(class_area.columns)
if missing:
    raise ValueError(f"Table_AreaStatistics_DakLak2024.csv missing columns: {missing}")

class_area["class_id"] = pd.to_numeric(class_area["class_id"], errors="coerce").astype(int)
class_area["area_ha"] = pd.to_numeric(class_area["area_ha"], errors="coerce")
class_area["class_name"] = class_area["class_id"].map(CLASS_INFO)

total_mapped_area = class_area["area_ha"].sum()
coffee_total = class_area.loc[class_area["class_id"].isin(COFFEE_CLASSES), "area_ha"].sum()

table5 = class_area.copy()
table5["land_cover_group"] = np.where(table5["class_id"].isin(COFFEE_CLASSES), "Coffee", "Non-coffee")
table5["share_of_mapped_area_percent"] = table5["area_ha"] / total_mapped_area * 100
table5["share_within_coffee_area_percent"] = np.where(
    table5["class_id"].isin(COFFEE_CLASSES) & (coffee_total > 0),
    table5["area_ha"] / coffee_total * 100,
    np.nan,
)
table5 = table5[[
    "class_id", "class_name", "land_cover_group", "area_ha",
    "share_of_mapped_area_percent", "share_within_coffee_area_percent",
]].sort_values("class_id")

table5.to_csv(TABLES_DIR / "Table5_MappedArea_By_LandCoverClass.csv", index=False, encoding="utf-8-sig")
try:
    with pd.ExcelWriter(TABLES_DIR / "Table5_MappedArea_By_LandCoverClass.xlsx", engine="openpyxl") as writer:
        table5.to_excel(writer, index=False, sheet_name="Table 5")
except Exception as e:
    print("Excel export skipped:", e)

display(table5)
print(f"Mapped coffee area: {coffee_total:,.0f} ha")
print("Saved:", TABLES_DIR / "Table5_MappedArea_By_LandCoverClass.csv")


In [ ]:

# ============================================================
# 0. USER CONFIGURATION
# ============================================================
from pathlib import Path

# Set this to the folder containing:
# - Table_TrainSamples_FullFeatureSpace_2024.csv
# - Table_ValSamples_FullFeatureSpace_2024.csv
INPUT_DIR = Path("data/raw")

# If the notebook is run inside /mnt/data or if files are placed beside the notebook,
# set INPUT_DIR = Path(".")
if not (INPUT_DIR / "Table_TrainSamples_FullFeatureSpace_2024.csv").exists():
    INPUT_DIR = Path("data")
if not (INPUT_DIR / "Table_TrainSamples_FullFeatureSpace_2024.csv").exists():
    INPUT_DIR = Path(".")

OUTPUT_DIR = SUPPLEMENTARY_DIR
TABLE_DIR = TABLES_DIR
FIG_DIR = FIGURES_DIR

# Main feature-selection parameters
FEATURE_COUNTS = [20, 25, 30, 35, 40]
MAIN_STRATEGY = "RF_Pearson"
PEARSON_THRESH = 0.90
VIF_THRESH = 10.0

# Random Forest parameters aligned with the final GEE/Python workflow
SEED = 2024
RANK_TREES = 250
RF_TREES = 2000
RF_MAX_FEATURES = 3
RF_MAX_SAMPLES = 0.65
RF_MIN_SAMPLES_LEAF = 1

# Repeated model-fitting seeds for sensitivity on the same train/validation split
# Increase to 10 if you want a stronger supplementary robustness check.
EVAL_SEEDS = [2024, 2025, 2026, 2027, 2028]

# Coffee production-system classes
COFFEE_CLASSES = [1, 2, 3]

CLASS_NAMES = {
    1: "Sun coffee",
    2: "Intercrop coffee",
    3: "Newly planted coffee",
    4: "Rubber",
    5: "Partially vegetative",
    6: "Rice",
    7: "Other upland crops",
    8: "Forest",
    9: "Water",
    10: "Built",
}

FEATURE_PREFIXES = ("S2_", "S1_", "L89_", "DEM_")

# If VIF calculation is too slow on your machine, set this True for quick tests.
FAST_DEBUG = False
if FAST_DEBUG:
    RANK_TREES = 80
    RF_TREES = 300
    EVAL_SEEDS = [2024]

INPUT_DIR, OUTPUT_DIR


In [5]:

# ============================================================
# 1. Imports and style
# ============================================================
import json
import math
import warnings
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    cohen_kappa_score,
    f1_score,
    confusion_matrix,
)
from sklearn.preprocessing import StandardScaler

plt.rcParams.update({
    "font.family": "Arial",
    "font.size": 10,
    "axes.labelsize": 10,
    "axes.titlesize": 11,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 9,
    "figure.titlesize": 12,
    "axes.linewidth": 0.8,
    "xtick.major.width": 0.8,
    "ytick.major.width": 0.8,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
})


In [6]:

# ============================================================
# 2. Utility functions: I/O, audit, markdown export
# ============================================================

def parse_gee_point_geo(value):
    try:
        g = json.loads(value)
        coords = g.get("coordinates")
        if isinstance(coords, list) and len(coords) >= 2:
            return float(coords[0]), float(coords[1])
    except Exception:
        pass
    return np.nan, np.nan


def add_lon_lat(df):
    df = df.copy()
    if "lon" in df.columns and "lat" in df.columns:
        return df
    if ".geo" in df.columns:
        coords = df[".geo"].apply(parse_gee_point_geo)
        df["lon"] = coords.apply(lambda x: x[0])
        df["lat"] = coords.apply(lambda x: x[1])
    return df


def detect_feature_columns(df):
    features = [
        c for c in df.columns
        if c.startswith(FEATURE_PREFIXES) and pd.api.types.is_numeric_dtype(df[c])
    ]
    return sorted(features)


def load_full97_samples(input_dir: Path):
    train_path = input_dir / "Table_TrainSamples_FullFeatureSpace_2024.csv"
    val_path = input_dir / "Table_ValSamples_FullFeatureSpace_2024.csv"

    if not train_path.exists():
        raise FileNotFoundError(f"Missing file: {train_path}")
    if not val_path.exists():
        raise FileNotFoundError(f"Missing file: {val_path}")

    train = pd.read_csv(train_path)
    val = pd.read_csv(val_path)
    train["split_origin"] = "train_full"
    val["split_origin"] = "val_full"

    train = add_lon_lat(train)
    val = add_lon_lat(val)

    df = pd.concat([train, val], ignore_index=True)

    if "class_id" not in df.columns:
        raise ValueError("Expected target column 'class_id' not found.")

    df = df.dropna(subset=["class_id"]).copy()
    df["class_id"] = df["class_id"].astype(int)

    feature_cols = detect_feature_columns(df)

    # Strict checks for the final workflow
    assert len(train) == 2100, f"Expected 2,100 training samples; got {len(train):,}"
    assert len(val) == 900, f"Expected 900 validation samples; got {len(val):,}"
    assert len(df) == 3000, f"Expected 3,000 samples; got {len(df):,}"
    assert len(feature_cols) == 97, f"Expected 97 predictors; got {len(feature_cols)}"

    class_counts = df["class_id"].value_counts().sort_index()
    assert sorted(class_counts.index.tolist()) == list(range(1, 11)), class_counts
    assert np.all(class_counts.values == 300), class_counts.to_dict()

    return train, val, df, feature_cols


def simple_markdown_table(df: pd.DataFrame) -> str:
    # Markdown export without requiring tabulate.
    d = df.copy()
    for col in d.columns:
        if isinstance(d[col].dtype, pd.CategoricalDtype):
            d[col] = d[col].astype(str)
    d = d.fillna("")
    cols = list(d.columns)
    lines = []
    lines.append("| " + " | ".join(map(str, cols)) + " |")
    lines.append("| " + " | ".join(["---"] * len(cols)) + " |")
    for _, row in d.iterrows():
        vals = [str(row[c]).replace("\n", " ") for c in cols]
        lines.append("| " + " | ".join(vals) + " |")
    return "\n".join(lines)


def save_table_bundle(df: pd.DataFrame, basename: str):
    csv_path = TABLE_DIR / f"{basename}.csv"
    md_path = TABLE_DIR / f"{basename}.md"
    tex_path = TABLE_DIR / f"{basename}.tex"

    df.to_csv(csv_path, index=False)
    md_path.write_text(simple_markdown_table(df), encoding="utf-8")
    tex_path.write_text(df.to_latex(index=False, escape=False), encoding="utf-8")

    return csv_path, md_path, tex_path


def mean_sd_text(mean, sd, decimals=3):
    if pd.isna(mean):
        return "NA"
    if pd.isna(sd):
        return f"{mean:.{decimals}f}"
    return f"{mean:.{decimals}f} ± {sd:.{decimals}f}"


def feature_group(feature: str) -> str:
    if feature.startswith("S2_"):
        return "Sentinel-2"
    if feature.startswith("S1_"):
        return "Sentinel-1"
    if feature.startswith("L89_"):
        return "Landsat 8/9"
    if feature.startswith("DEM_"):
        return "DEM"
    return "Other"


In [7]:

# ============================================================
# 3. Load data and run input audit
# ============================================================
train_df, val_df, all_df, feature_cols = load_full97_samples(INPUT_DIR)

X_train_raw = train_df[feature_cols].copy()
y_train = train_df["class_id"].astype(int).values

X_val_raw = val_df[feature_cols].copy()
y_val = val_df["class_id"].astype(int).values

# Fit preprocessing only on training set to avoid leakage.
imputer = SimpleImputer(strategy="median")
X_train = pd.DataFrame(imputer.fit_transform(X_train_raw), columns=feature_cols, index=train_df.index)
X_val = pd.DataFrame(imputer.transform(X_val_raw), columns=feature_cols, index=val_df.index)

# Audit tables
audit_class = all_df["class_id"].value_counts().sort_index().rename_axis("class_id").reset_index(name="n")
audit_class["class_name"] = audit_class["class_id"].map(CLASS_NAMES)

audit_features = pd.DataFrame({
    "feature": feature_cols,
    "source_group": [feature_group(f) for f in feature_cols],
    "missing_train": X_train_raw[feature_cols].isna().sum().values,
    "missing_val": X_val_raw[feature_cols].isna().sum().values,
    "zero_train": (X_train_raw[feature_cols] == 0).sum().values,
    "zero_val": (X_val_raw[feature_cols] == 0).sum().values,
})

save_table_bundle(audit_class, "InputAudit_ClassCounts")
save_table_bundle(audit_features, "InputAudit_FeatureMissingZero")

print(f"Training samples: {len(train_df):,}")
print(f"Validation samples: {len(val_df):,}")
print(f"Total samples: {len(all_df):,}")
print(f"Predictors: {len(feature_cols)}")
print(audit_class.to_string(index=False))
audit_features.groupby("source_group").size()


Training samples: 2,100
Validation samples: 900
Total samples: 3,000
Predictors: 97
 class_id   n           class_name
        1 300           Sun coffee
        2 300     Intercrop coffee
        3 300 Newly planted coffee
        4 300               Rubber
        5 300 Partially vegetative
        6 300                 Rice
        7 300   Other upland crops
        8 300               Forest
        9 300                Water
       10 300                Built


source_group
DEM             4
Landsat 8/9    26
Sentinel-1     15
Sentinel-2     52
dtype: int64


## Feature-selection functions

All feature selection is performed on the **training set only**. The validation set is used only for model evaluation.


In [8]:

# ============================================================
# 4. Feature ranking and filtering strategies
# ============================================================

def rf_rank_features(X: pd.DataFrame, y: np.ndarray, features: List[str], seed: int = SEED) -> pd.DataFrame:
    rf = RandomForestClassifier(
        n_estimators=RANK_TREES,
        max_features=min(RF_MAX_FEATURES, len(features)),
        bootstrap=True,
        max_samples=RF_MAX_SAMPLES,
        min_samples_leaf=RF_MIN_SAMPLES_LEAF,
        random_state=seed,
        n_jobs=-1,
    )
    rf.fit(X[features], y)
    out = pd.DataFrame({
        "feature": features,
        "rf_rank_importance": rf.feature_importances_,
    }).sort_values("rf_rank_importance", ascending=False).reset_index(drop=True)
    out["rf_rank"] = np.arange(1, len(out) + 1)
    return out


def greedy_pearson_filter(X: pd.DataFrame, ranked_features: List[str], threshold: float, max_features: int) -> List[str]:
    selected = []
    corr = X[ranked_features].corr(method="pearson").abs()
    for feat in ranked_features:
        if len(selected) >= max_features:
            break
        if not selected:
            selected.append(feat)
        else:
            max_corr = corr.loc[feat, selected].max()
            if pd.isna(max_corr) or max_corr < threshold:
                selected.append(feat)
    return selected


def compute_vif_values(X_np: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    """Compute VIF for each column using least squares. Assumes X_np has no NaN and is standardized."""
    n_features = X_np.shape[1]
    vifs = np.empty(n_features, dtype=float)
    for j in range(n_features):
        yj = X_np[:, j]
        others = np.delete(X_np, j, axis=1)
        # Add intercept for numerical stability
        A = np.column_stack([np.ones(others.shape[0]), others])
        try:
            beta, *_ = np.linalg.lstsq(A, yj, rcond=None)
            pred = A @ beta
            ss_res = np.sum((yj - pred) ** 2)
            ss_tot = np.sum((yj - np.mean(yj)) ** 2)
            r2 = 1.0 - ss_res / ss_tot if ss_tot > eps else 0.0
            r2 = min(max(r2, 0.0), 1.0 - eps)
            vifs[j] = 1.0 / (1.0 - r2)
        except Exception:
            vifs[j] = np.inf
    return vifs


def iterative_vif_filter(X: pd.DataFrame, features: List[str], threshold: float = 10.0, max_iter: int = 200):
    """Maskell-style iterative VIF reduction. Unsupervised; uses training features only."""
    remaining = list(features)
    history = []

    scaler = StandardScaler()
    X_scaled_all = pd.DataFrame(
        scaler.fit_transform(X[features]),
        columns=features,
        index=X.index
    )

    for step in range(max_iter):
        if len(remaining) <= 2:
            break

        X_np = X_scaled_all[remaining].values
        vifs = compute_vif_values(X_np)
        vif_df = pd.DataFrame({"feature": remaining, "vif": vifs}).sort_values("vif", ascending=False)
        max_vif = float(vif_df.iloc[0]["vif"])
        worst_feature = str(vif_df.iloc[0]["feature"])

        history.append({
            "step": step + 1,
            "n_features_before": len(remaining),
            "removed_feature": worst_feature if max_vif > threshold else "",
            "max_vif": max_vif,
        })

        if max_vif <= threshold:
            break

        remaining.remove(worst_feature)

    final_vifs = pd.DataFrame({
        "feature": remaining,
        "vif": compute_vif_values(X_scaled_all[remaining].values) if len(remaining) > 1 else [1.0],
    }).sort_values("vif", ascending=False)

    return remaining, pd.DataFrame(history), final_vifs


def greedy_vif_filter_ranked(X: pd.DataFrame, ranked_features: List[str], threshold: float, max_features: int) -> List[str]:
    """Hybrid supervised + VIF strategy: add RF-ranked features if selected-set VIF remains <= threshold."""
    selected = []
    scaler = StandardScaler()
    X_scaled = pd.DataFrame(scaler.fit_transform(X[ranked_features]), columns=ranked_features, index=X.index)

    for feat in ranked_features:
        if len(selected) >= max_features:
            break

        candidate = selected + [feat]
        if len(candidate) <= 2:
            selected.append(feat)
            continue

        vifs = compute_vif_values(X_scaled[candidate].values)
        max_vif = np.nanmax(vifs)

        if np.isfinite(max_vif) and max_vif <= threshold:
            selected.append(feat)

    return selected


def build_selection_sets(X: pd.DataFrame, y: np.ndarray, feature_cols: List[str]):
    ranking = rf_rank_features(X, y, feature_cols, seed=SEED)
    ranked = ranking["feature"].tolist()

    selected_sets = []

    # 1) RF rank only
    for n in FEATURE_COUNTS:
        selected_sets.append({
            "strategy": "RF_rank_only",
            "feature_count_target": n,
            "selected_features": ranked[:n],
            "selection_note": "Top-N predictors by RF ranking only",
        })

    # 2) RF + Pearson |r| threshold
    for n in FEATURE_COUNTS:
        selected_sets.append({
            "strategy": "RF_Pearson",
            "feature_count_target": n,
            "selected_features": greedy_pearson_filter(X, ranked, PEARSON_THRESH, n),
            "selection_note": f"RF ranking + Pearson |r| < {PEARSON_THRESH}",
        })

    # 3) RF + VIF threshold
    for n in FEATURE_COUNTS:
        selected_sets.append({
            "strategy": "RF_VIF10",
            "feature_count_target": n,
            "selected_features": greedy_vif_filter_ranked(X, ranked, VIF_THRESH, n),
            "selection_note": f"RF ranking + greedy selected-set VIF ≤ {VIF_THRESH}",
        })

    # 4) VIF-only Maskell-style comparator
    vif_features, vif_history, vif_final = iterative_vif_filter(X, feature_cols, threshold=VIF_THRESH)
    selected_sets.append({
        "strategy": "VIF10_only",
        "feature_count_target": len(vif_features),
        "selected_features": vif_features,
        "selection_note": f"Iterative VIF-only filter with VIF ≤ {VIF_THRESH}",
    })

    return selected_sets, ranking, vif_history, vif_final


selected_sets, rf_ranking, vif_history, vif_final = build_selection_sets(X_train, y_train, feature_cols)

rf_ranking.to_csv(TABLE_DIR / "RF_Ranking_Full97_TrainingOnly.csv", index=False)
vif_history.to_csv(TABLE_DIR / "VIF10_IterativeRemoval_History.csv", index=False)
vif_final.to_csv(TABLE_DIR / "VIF10_FinalVIFs.csv", index=False)

selection_overview = pd.DataFrame([
    {
        "strategy": s["strategy"],
        "feature_count_target": s["feature_count_target"],
        "n_selected": len(s["selected_features"]),
        "selection_note": s["selection_note"],
        "selected_features": "; ".join(s["selected_features"]),
    }
    for s in selected_sets
])
save_table_bundle(selection_overview, "SelectionOverview_AllStrategies")
selection_overview[["strategy", "feature_count_target", "n_selected", "selection_note"]]


,strategy,feature_count_target,n_selected,selection_note
0,RF_rank_only,20,20,Top-N predictors by RF ranking only
1,RF_rank_only,25,25,Top-N predictors by RF ranking only
2,RF_rank_only,30,30,Top-N predictors by RF ranking only
3,RF_rank_only,35,35,Top-N predictors by RF ranking only
4,RF_rank_only,40,40,Top-N predictors by RF ranking only
5,RF_Pearson,20,20,RF ranking + Pearson |r| < 0.9
6,RF_Pearson,25,25,RF ranking + Pearson |r| < 0.9
7,RF_Pearson,30,30,RF ranking + Pearson |r| < 0.9
8,RF_Pearson,35,35,RF ranking + Pearson |r| < 0.9
9,RF_Pearson,40,40,RF ranking + Pearson |r| < 0.9



## Model evaluation

Each selected feature set is evaluated on the independent 900-point validation table.  
The Random Forest is re-fit with multiple seeds to report mean ± SD sensitivity.


In [9]:

# ============================================================
# 5. Model evaluation
# ============================================================

def evaluate_predictions(y_true, y_pred):
    return {
        "OA": accuracy_score(y_true, y_pred),
        "Kappa": cohen_kappa_score(y_true, y_pred),
        "MacroF1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "CoffeeSubclassMacroF1": f1_score(
            y_true, y_pred,
            labels=COFFEE_CLASSES,
            average="macro",
            zero_division=0,
        ),
    }


def classwise_f1(y_true, y_pred):
    labels = list(range(1, 11))
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    rowsum = cm.sum(axis=1)
    colsum = cm.sum(axis=0)
    diag = np.diag(cm)

    rows = []
    for i, lab in enumerate(labels):
        pa = diag[i] / rowsum[i] if rowsum[i] > 0 else np.nan
        ua = diag[i] / colsum[i] if colsum[i] > 0 else np.nan
        f1 = 2 * pa * ua / (pa + ua) if (pa + ua) > 0 else np.nan
        rows.append({
            "class_id": lab,
            "class_name": CLASS_NAMES[lab],
            "producer_accuracy": pa,
            "user_accuracy": ua,
            "f1": f1,
            "support_true": rowsum[i],
            "support_pred": colsum[i],
        })
    return pd.DataFrame(rows)


def fit_predict_rf(X_train, y_train, X_val, features, seed):
    rf = RandomForestClassifier(
        n_estimators=RF_TREES,
        max_features=min(RF_MAX_FEATURES, len(features)),
        bootstrap=True,
        max_samples=RF_MAX_SAMPLES,
        min_samples_leaf=RF_MIN_SAMPLES_LEAF,
        random_state=seed,
        n_jobs=-1,
    )
    rf.fit(X_train[features], y_train)
    pred = rf.predict(X_val[features])
    return pred, rf


all_metrics = []
all_classwise = []
all_selected_features = []
all_importances = []

for s in selected_sets:
    strategy = s["strategy"]
    n_target = s["feature_count_target"]
    features = s["selected_features"]
    n_selected = len(features)

    for seed in EVAL_SEEDS:
        pred, rf_model = fit_predict_rf(X_train, y_train, X_val, features, seed=seed)
        metrics = evaluate_predictions(y_val, pred)
        metrics.update({
            "strategy": strategy,
            "feature_count_target": n_target,
            "n_selected": n_selected,
            "seed": seed,
        })
        all_metrics.append(metrics)

        cls = classwise_f1(y_val, pred)
        cls.insert(0, "seed", seed)
        cls.insert(0, "n_selected", n_selected)
        cls.insert(0, "feature_count_target", n_target)
        cls.insert(0, "strategy", strategy)
        all_classwise.append(cls)

        imp = pd.DataFrame({
            "feature": features,
            "importance": rf_model.feature_importances_,
            "strategy": strategy,
            "feature_count_target": n_target,
            "n_selected": n_selected,
            "seed": seed,
        }).sort_values("importance", ascending=False)
        imp["rank"] = np.arange(1, len(imp) + 1)
        all_importances.append(imp)

    for rank, feat in enumerate(features, start=1):
        all_selected_features.append({
            "strategy": strategy,
            "feature_count_target": n_target,
            "n_selected": n_selected,
            "rank_selected": rank,
            "feature": feat,
            "source_group": feature_group(feat),
        })

metrics_df = pd.DataFrame(all_metrics)
classwise_df = pd.concat(all_classwise, ignore_index=True)
selected_features_df = pd.DataFrame(all_selected_features)
importance_df = pd.concat(all_importances, ignore_index=True)

metrics_df.to_csv(TABLE_DIR / "RawMetrics_AllStrategies_AllSeeds.csv", index=False)
classwise_df.to_csv(TABLE_DIR / "RawClasswiseMetrics_AllStrategies_AllSeeds.csv", index=False)
selected_features_df.to_csv(TABLE_DIR / "RawSelectedFeatures_AllStrategies.csv", index=False)
importance_df.to_csv(TABLE_DIR / "RawRFImportance_AllStrategies_AllSeeds.csv", index=False)

metrics_df.head()


,OA,Kappa,MacroF1,CoffeeSubclassMacroF1,strategy,feature_count_target,n_selected,seed
0,0.923333,0.914815,0.923322,0.863809,RF_rank_only,20,20,2024
1,0.922222,0.913580,0.922252,0.863845,RF_rank_only,20,20,2025
2,0.923333,0.914815,0.923330,0.865324,RF_rank_only,20,20,2026
3,0.922222,0.913580,0.922273,0.863809,RF_rank_only,20,20,2027
4,0.922222,0.913580,0.922275,0.862067,RF_rank_only,20,20,2028


In [10]:

# ============================================================
# 6. Summarize metrics and choose recommended Top-N
# ============================================================

summary_metrics = (
    metrics_df
    .groupby(["strategy", "feature_count_target", "n_selected"], as_index=False)
    .agg(
        n_runs=("seed", "count"),
        OA_mean=("OA", "mean"), OA_sd=("OA", "std"),
        Kappa_mean=("Kappa", "mean"), Kappa_sd=("Kappa", "std"),
        MacroF1_mean=("MacroF1", "mean"), MacroF1_sd=("MacroF1", "std"),
        CoffeeSubclassMacroF1_mean=("CoffeeSubclassMacroF1", "mean"),
        CoffeeSubclassMacroF1_sd=("CoffeeSubclassMacroF1", "std"),
    )
)

# One-standard-error style rule for the main RF_Pearson Top-N choice.
main = summary_metrics[summary_metrics["strategy"] == MAIN_STRATEGY].copy()
best_idx = main["CoffeeSubclassMacroF1_mean"].idxmax()
best_row = main.loc[best_idx]
best_mean = best_row["CoffeeSubclassMacroF1_mean"]
best_sd = best_row["CoffeeSubclassMacroF1_sd"]
threshold = best_mean - best_sd if not pd.isna(best_sd) else best_mean

eligible = main[main["CoffeeSubclassMacroF1_mean"] >= threshold].sort_values("n_selected")
recommended_row = eligible.iloc[0]

recommendation = {
    "main_strategy": MAIN_STRATEGY,
    "best_feature_count_by_mean": int(best_row["n_selected"]),
    "best_coffee_subclass_macro_f1_mean": float(best_mean),
    "best_coffee_subclass_macro_f1_sd": float(best_sd) if not pd.isna(best_sd) else np.nan,
    "one_sd_threshold": float(threshold),
    "recommended_feature_count_one_sd_rule": int(recommended_row["n_selected"]),
    "recommended_coffee_subclass_macro_f1_mean": float(recommended_row["CoffeeSubclassMacroF1_mean"]),
    "recommended_oa_mean": float(recommended_row["OA_mean"]),
}

pd.DataFrame([recommendation]).to_csv(TABLE_DIR / "RecommendedFeatureCount_OneSDRule.csv", index=False)

summary_metrics_sorted = summary_metrics.sort_values(["strategy", "feature_count_target"])
save_table_bundle(summary_metrics_sorted, "SummaryMetrics_AllStrategies")
summary_metrics_sorted


,strategy,feature_count_target,n_selected,n_runs,OA_mean,OA_sd,Kappa_mean,Kappa_sd,MacroF1_mean,MacroF1_sd,CoffeeSubclassMacroF1_mean,CoffeeSubclassMacroF1_sd
0,RF_Pearson,20,20,5,0.921556,0.001267,0.912840,0.001408,0.921375,0.001278,0.871582,0.002906
1,RF_Pearson,25,25,5,0.928000,0.002767,0.920000,0.003074,0.927869,0.002789,0.882606,0.003657
2,RF_Pearson,30,30,5,0.931778,0.002018,0.924198,0.002243,0.931746,0.002060,0.889431,0.002986
3,RF_Pearson,35,35,5,0.935333,0.000930,0.928148,0.001033,0.935286,0.000928,0.889421,0.000867
4,RF_Pearson,40,40,5,0.936222,0.001267,0.929136,0.001408,0.936157,0.001282,0.889980,0.003790
5,RF_VIF10,20,15,5,0.924889,0.001267,0.916543,0.001408,0.924842,0.001179,0.865786,0.003932
6,RF_VIF10,25,15,5,0.924889,0.001267,0.916543,0.001408,0.924842,0.001179,0.865786,0.003932
7,RF_VIF10,30,15,5,0.924889,0.001267,0.916543,0.001408,0.924842,0.001179,0.865786,0.003932
8,RF_VIF10,35,15,5,0.924889,0.001267,0.916543,0.001408,0.924842,0.001179,0.865786,0.003932
9,RF_VIF10,40,15,5,0.924889,0.001267,0.916543,0.001408,0.924842,0.001179,0.865786,0.003932


In [ ]:
# ============================================================
# 7a+b. Statistical comparison among feature-selection strategies
# ============================================================

from scipy.stats import friedmanchisquare, wilcoxon
from itertools import combinations
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. Define key strategies for comparison
# ------------------------------------------------------------
# These are the same strategies shown in Supplementary Figure S6.
strategy_order = ["RF_Pearson", "RF_VIF10", "RF_rank_only", "VIF10_only"]

strategy_label_map = {
    "RF_Pearson": "RF_Pearson",
    "RF_VIF10": "RF_VIF10",
    "RF_rank_only": "RF_rank_only",
    "VIF10_only": "VIF10_only",
}

metric_cols = {
    "OA": "Overall accuracy",
    "MacroF1": "Macro F1",
    "CoffeeSubclassMacroF1": "Coffee F1",
}

# Keep only strategies used in Figure Sy.
# For RF_Pearson, RF_VIF10, and RF_rank_only, use Top-25 target.
# For VIF10_only, use the VIF-selected set.
strategy_stats_df = metrics_df[
    (
        (metrics_df["strategy"].isin(["RF_Pearson", "RF_VIF10", "RF_rank_only"])) &
        (metrics_df["feature_count_target"] == 25)
    ) |
    (metrics_df["strategy"] == "VIF10_only")
].copy()

# Make sure strategy order is stable.
strategy_stats_df["strategy"] = pd.Categorical(
    strategy_stats_df["strategy"],
    categories=strategy_order,
    ordered=True
)

# ------------------------------------------------------------
# 2. Summary table: mean, SD, SE, 95% CI
# ------------------------------------------------------------
summary_stats = (
    strategy_stats_df
    .groupby(["strategy", "feature_count_target", "n_selected"], observed=True)
    .agg(
        n_runs=("seed", "count"),
        OA_mean=("OA", "mean"),
        OA_sd=("OA", "std"),
        MacroF1_mean=("MacroF1", "mean"),
        MacroF1_sd=("MacroF1", "std"),
        CoffeeF1_mean=("CoffeeSubclassMacroF1", "mean"),
        CoffeeF1_sd=("CoffeeSubclassMacroF1", "std"),
    )
    .reset_index()
)

for prefix in ["OA", "MacroF1", "CoffeeF1"]:
    summary_stats[f"{prefix}_se"] = summary_stats[f"{prefix}_sd"] / np.sqrt(summary_stats["n_runs"])
    summary_stats[f"{prefix}_ci95"] = 1.96 * summary_stats[f"{prefix}_se"]

summary_stats["strategy_label"] = summary_stats.apply(
    lambda r: f"{r['strategy']}\n(n={int(r['n_selected'])})",
    axis=1
)

summary_stats = summary_stats.sort_values("strategy")

save_table_bundle(
    summary_stats,
    "Detail_FeatureSelection_StrategySummary_CI95"
)

summary_stats

In [12]:
# ============================================================
# 7c. Friedman + pairwise Wilcoxon signed-rank tests
# ============================================================

def holm_adjust(p_values):
    """
    Holm-Bonferroni correction.
    Returns adjusted p-values in the original order.
    """
    p_values = np.asarray(p_values, dtype=float)
    n = len(p_values)
    order = np.argsort(p_values)
    adjusted = np.empty(n, dtype=float)

    prev = 0
    for rank, idx in enumerate(order):
        adj = (n - rank) * p_values[idx]
        adj = max(adj, prev)
        adjusted[idx] = min(adj, 1.0)
        prev = adjusted[idx]

    return adjusted


friedman_rows = []
pairwise_rows = []

for metric, metric_label in metric_cols.items():

    # Pivot to paired format:
    # rows = seed/repeat unit, columns = strategy
    wide = strategy_stats_df.pivot_table(
        index="seed",
        columns="strategy",
        values=metric,
        aggfunc="mean",
        observed=True
    )

    # Keep only seeds where all strategies are available.
    wide = wide.dropna(subset=strategy_order)

    # Friedman global test across strategies
    if len(wide) >= 3:
        stat, p_value = friedmanchisquare(
            *[wide[s].values for s in strategy_order]
        )
    else:
        stat, p_value = np.nan, np.nan

    friedman_rows.append({
        "metric": metric_label,
        "test": "Friedman test",
        "n_paired_units": len(wide),
        "statistic": stat,
        "p_value": p_value,
    })

    # Pairwise Wilcoxon signed-rank tests
    raw_rows = []
    raw_p = []

    for s1, s2 in combinations(strategy_order, 2):
        x = wide[s1].values
        y = wide[s2].values

        if len(x) >= 3 and not np.allclose(x, y):
            try:
                w_stat, p_pair = wilcoxon(x, y, zero_method="wilcox", alternative="two-sided")
            except ValueError:
                w_stat, p_pair = np.nan, np.nan
        else:
            w_stat, p_pair = np.nan, np.nan

        mean_diff = np.nanmean(x - y)

        raw_rows.append({
            "metric": metric_label,
            "comparison": f"{s1} vs {s2}",
            "strategy_1": s1,
            "strategy_2": s2,
            "n_paired_units": len(wide),
            "mean_difference_strategy1_minus_strategy2": mean_diff,
            "wilcoxon_statistic": w_stat,
            "p_value_raw": p_pair,
        })
        raw_p.append(p_pair)

    # Holm correction per metric
    valid_mask = ~pd.isna(raw_p)
    raw_p_array = np.asarray(raw_p, dtype=float)
    adj_p = np.full(len(raw_p_array), np.nan)

    if valid_mask.sum() > 0:
        adj_p[valid_mask] = holm_adjust(raw_p_array[valid_mask])

    for row, p_adj in zip(raw_rows, adj_p):
        row["p_value_holm"] = p_adj
        row["significance_holm"] = (
            "***" if pd.notna(p_adj) and p_adj < 0.001 else
            "**" if pd.notna(p_adj) and p_adj < 0.01 else
            "*" if pd.notna(p_adj) and p_adj < 0.05 else
            "ns" if pd.notna(p_adj) else "NA"
        )
        pairwise_rows.append(row)

friedman_df = pd.DataFrame(friedman_rows)
pairwise_df = pd.DataFrame(pairwise_rows)

save_table_bundle(
    friedman_df,
    "Detail_FeatureSelection_FriedmanGlobalTests"
)

save_table_bundle(
    pairwise_df,
    "Detail_FeatureSelection_WilcoxonHolmPairwise"
)

# Also export both into one Excel workbook
stats_xlsx = TABLE_DIR / "Supplementary_StatisticalComparison_FeatureSelectionStrategies.xlsx"

with pd.ExcelWriter(stats_xlsx, engine="openpyxl") as writer:
    summary_stats.to_excel(writer, sheet_name="Mean_SD_SE_CI95", index=False)
    friedman_df.to_excel(writer, sheet_name="Friedman_tests", index=False)
    pairwise_df.to_excel(writer, sheet_name="Wilcoxon_Holm", index=False)

friedman_df, pairwise_df.head(10)

(             metric           test  n_paired_units  statistic   p_value
 0  Overall accuracy  Friedman test               5  12.326087  0.006346
 1          Macro F1  Friedman test               5  10.680000  0.013588
 2         Coffee F1  Friedman test               5  13.560000  0.003570,
              metric                  comparison    strategy_1    strategy_2  \
 0  Overall accuracy      RF_Pearson vs RF_VIF10    RF_Pearson      RF_VIF10   
 1  Overall accuracy  RF_Pearson vs RF_rank_only    RF_Pearson  RF_rank_only   
 2  Overall accuracy    RF_Pearson vs VIF10_only    RF_Pearson    VIF10_only   
 3  Overall accuracy    RF_VIF10 vs RF_rank_only      RF_VIF10  RF_rank_only   
 4  Overall accuracy      RF_VIF10 vs VIF10_only      RF_VIF10    VIF10_only   
 5  Overall accuracy  RF_rank_only vs VIF10_only  RF_rank_only    VIF10_only   
 6          Macro F1      RF_Pearson vs RF_VIF10    RF_Pearson      RF_VIF10   
 7          Macro F1  RF_Pearson vs RF_rank_only    RF_Pearson  RF_


## Supplementary tables

The next cell exports manuscript-ready supplementary tables:

- **Supplementary Table Sx.** Feature-count sensitivity for RF ranking + Pearson filtering.
- **Supplementary Table Sy.** Feature-selection strategy comparison.
- **Supplementary Table Sz.** Class-wise F1 for key strategies.
- **Supplementary Table Sa.** Selected features and sensor groups for key strategies.


In [13]:

# ============================================================
# 7. Supplementary tables
# ============================================================

# Table Sx: Feature-count sensitivity for main strategy
Sx_base = summary_metrics_sorted[summary_metrics_sorted["strategy"] == "RF_Pearson"].copy()
Sx = pd.DataFrame({
    "Strategy": Sx_base["strategy"],
    "Target feature count": Sx_base["feature_count_target"],
    "Selected predictors": Sx_base["n_selected"],
    "Overall accuracy": [mean_sd_text(m, s) for m, s in zip(Sx_base["OA_mean"], Sx_base["OA_sd"])],
    "Kappa": [mean_sd_text(m, s) for m, s in zip(Sx_base["Kappa_mean"], Sx_base["Kappa_sd"])],
    "Macro F1": [mean_sd_text(m, s) for m, s in zip(Sx_base["MacroF1_mean"], Sx_base["MacroF1_sd"])],
    "Coffee-subclass macro F1": [mean_sd_text(m, s) for m, s in zip(Sx_base["CoffeeSubclassMacroF1_mean"], Sx_base["CoffeeSubclassMacroF1_sd"])],
})

# Table Sy: Strategy comparison focusing on main Top25 and VIF alternatives
strategy_keep = (
    ((summary_metrics_sorted["strategy"] == "RF_Pearson") & (summary_metrics_sorted["feature_count_target"] == 25)) |
    ((summary_metrics_sorted["strategy"] == "RF_rank_only") & (summary_metrics_sorted["feature_count_target"] == 25)) |
    ((summary_metrics_sorted["strategy"] == "RF_VIF10") & (summary_metrics_sorted["feature_count_target"] == 25)) |
    (summary_metrics_sorted["strategy"] == "VIF10_only")
)
Sy_base = summary_metrics_sorted[strategy_keep].copy()
Sy = pd.DataFrame({
    "Strategy": Sy_base["strategy"],
    "Target feature count": Sy_base["feature_count_target"],
    "Selected predictors": Sy_base["n_selected"],
    "Overall accuracy": [mean_sd_text(m, s) for m, s in zip(Sy_base["OA_mean"], Sy_base["OA_sd"])],
    "Kappa": [mean_sd_text(m, s) for m, s in zip(Sy_base["Kappa_mean"], Sy_base["Kappa_sd"])],
    "Macro F1": [mean_sd_text(m, s) for m, s in zip(Sy_base["MacroF1_mean"], Sy_base["MacroF1_sd"])],
    "Coffee-subclass macro F1": [mean_sd_text(m, s) for m, s in zip(Sy_base["CoffeeSubclassMacroF1_mean"], Sy_base["CoffeeSubclassMacroF1_sd"])],
})

# Table Sz: Class-wise F1 for key strategies
class_summary = (
    classwise_df
    .groupby(["strategy", "feature_count_target", "n_selected", "class_id", "class_name"], as_index=False)
    .agg(f1_mean=("f1", "mean"), f1_sd=("f1", "std"))
)
key_class = class_summary[
    ((class_summary["strategy"] == "RF_Pearson") & (class_summary["feature_count_target"] == 25)) |
    ((class_summary["strategy"] == "RF_VIF10") & (class_summary["feature_count_target"] == 25)) |
    (class_summary["strategy"] == "VIF10_only")
].copy()
key_class["F1 (mean ± SD)"] = [mean_sd_text(m, s) for m, s in zip(key_class["f1_mean"], key_class["f1_sd"])]
Sz = key_class.pivot_table(
    index=["class_id", "class_name"],
    columns="strategy",
    values="F1 (mean ± SD)",
    aggfunc="first"
).reset_index().rename(columns={"class_id": "Class ID", "class_name": "Class"})

# Table Sa: Selected features for key strategies
selected_key = selected_features_df[
    ((selected_features_df["strategy"] == "RF_Pearson") & (selected_features_df["feature_count_target"] == 25)) |
    ((selected_features_df["strategy"] == "RF_VIF10") & (selected_features_df["feature_count_target"] == 25)) |
    (selected_features_df["strategy"] == "VIF10_only")
].copy()
Sa = selected_key[["strategy", "feature_count_target", "rank_selected", "feature", "source_group"]].rename(columns={
    "strategy": "Strategy",
    "feature_count_target": "Target feature count",
    "rank_selected": "Rank",
    "feature": "Predictor",
    "source_group": "Source group",
})

save_table_bundle(Sx, "Detail_FeatureSelection_CountSensitivity_RawRun")
save_table_bundle(Sy, "Detail_FeatureSelection_StrategyComparison_RawRun")
save_table_bundle(Sz, "Detail_FeatureSelection_ClasswiseF1_ByStrategy")
save_table_bundle(Sa, "Detail_FeatureSelection_SelectedFeatures_ByStrategy")

# Excel workbook
xlsx_path = TABLE_DIR / "Supplementary_Tables_FeatureSelectionSensitivity_DakLak2024.xlsx"
with pd.ExcelWriter(xlsx_path, engine="openpyxl") as writer:
    Sx.to_excel(writer, sheet_name="Sx_count_sensitivity", index=False)
    Sy.to_excel(writer, sheet_name="Sy_strategy_comparison", index=False)
    Sz.to_excel(writer, sheet_name="Sz_classwise_F1", index=False)
    Sa.to_excel(writer, sheet_name="Sa_selected_features", index=False)
    summary_metrics_sorted.to_excel(writer, sheet_name="All_metrics", index=False)
    selection_overview.to_excel(writer, sheet_name="All_selected_sets", index=False)
    vif_history.to_excel(writer, sheet_name="VIF_history", index=False)
    vif_final.to_excel(writer, sheet_name="VIF_final", index=False)

Sx


,Strategy,Target feature count,Selected predictors,Overall accuracy,Kappa,Macro F1,Coffee-subclass macro F1
0,RF_Pearson,20,20,0.922 ± 0.001,0.913 ± 0.001,0.921 ± 0.001,0.872 ± 0.003
1,RF_Pearson,25,25,0.928 ± 0.003,0.920 ± 0.003,0.928 ± 0.003,0.883 ± 0.004
2,RF_Pearson,30,30,0.932 ± 0.002,0.924 ± 0.002,0.932 ± 0.002,0.889 ± 0.003
3,RF_Pearson,35,35,0.935 ± 0.001,0.928 ± 0.001,0.935 ± 0.001,0.889 ± 0.001
4,RF_Pearson,40,40,0.936 ± 0.001,0.929 ± 0.001,0.936 ± 0.001,0.890 ± 0.004



## Supplementary figures

The next cell exports two publication-style supplementary figures:

1. **Supplementary Figure S5.** Feature-count sensitivity for the main RF + Pearson strategy.
2. **Supplementary Figure S6.** Strategy comparison and sensor composition for Top-25/VIF alternatives.


In [ ]:
# ============================================================
# 8. Supplementary figures (S5, S6)
# ============================================================
# The plotting code used to live here, duplicated in
# results/figures/regenerate_S5_S6_feature_selection_figures.py.
# Consolidated 2026-08-09 to a single source of truth: that
# standalone script is now the only place this plotting logic
# lives, to avoid the two copies drifting apart. It reads the raw
# per-seed CSVs already saved earlier in this notebook
# (RawMetrics_AllStrategies_AllSeeds.csv,
# RawClasswiseMetrics_AllStrategies_AllSeeds.csv,
# RawSelectedFeatures_AllStrategies.csv), so it does not require
# re-running any RF fitting -- run it after this notebook:
#
#   python results/figures/regenerate_S5_S6_feature_selection_figures.py
#
# Outputs: Supplementary_Figure_S5_FeatureCountSensitivity.png/.pdf,
#          Supplementary_Figure_S6_StrategyComparison_SensorComposition.png/.pdf


In [ ]:

# ============================================================
# 9. Manuscript-ready notes and captions
# ============================================================

caption_text = f"""
Supplementary Table Sx. Feature-count sensitivity of the RF-ranking plus Pearson-correlation selected-feature strategy.
The table compares Top 20, 25, 30, 35 and 40 predictors selected using Random Forest ranking followed by Pearson-correlation filtering with |r| < {PEARSON_THRESH}. Values are mean ± SD across {len(EVAL_SEEDS)} Random Forest fitting seeds.

Supplementary Table Sy. Feature-selection strategy comparison.
The table compares the main RF + Pearson Top-25 strategy with RF-ranking only, RF + VIF, and VIF-only alternatives. VIF-based filtering used a threshold of VIF ≤ {VIF_THRESH} and was included as a supplementary sensitivity analysis following the logic of previous coffee-mapping work in Dak Lak.

Supplementary Table Sz. Class-wise F1 for key feature-selection strategies.
The table reports class-wise F1-scores for the main RF + Pearson Top-25 strategy and VIF-based alternatives.

Supplementary Table Sa. Selected predictors for key feature-selection strategies.
The table lists predictors retained by the main RF + Pearson Top-25 strategy and the VIF-based alternatives, together with their sensor/source group.

Supplementary Figure S5. Feature-count sensitivity of the RF + Pearson selected-feature model.
(A) Overall accuracy, macro F1 and coffee-subclass macro F1 across Top 20, 25, 30, 35 and 40 selected predictors. The dashed vertical line indicates the final Top-25 model. (B) Class-wise F1 for the three coffee production systems across feature counts.

Supplementary Figure S6. Feature-selection strategy comparison and sensor composition.
(A) Overall accuracy, macro F1 and coffee-subclass macro F1 for RF-ranking only, RF + Pearson, RF + VIF and VIF-only strategies. (B) Sensor composition of the selected predictors for the same strategies.
""".strip()

(OUTPUT_DIR / "Supplementary_Captions_FeatureSelectionSensitivity.txt").write_text(caption_text, encoding="utf-8")

methods_text = f"""
Suggested Methods text:

Feature-selection sensitivity analysis was conducted using the exported FULL97 training and validation sample tables. The main strategy ranked predictors using Random Forest Gini importance on the training samples and then applied greedy Pearson-correlation filtering (|r| < {PEARSON_THRESH}) to reduce redundancy. To test whether the final Top-25 predictor set was sensitive to the number of retained predictors, we evaluated Top 20, 25, 30, 35 and 40 selected-feature models. As a supplementary comparison with previous coffee-mapping work in Dak Lak, we also tested a variance inflation factor (VIF)-based filter. VIF was calculated on the training samples only, and predictors were iteratively removed until all remaining predictors had VIF ≤ {VIF_THRESH}. A hybrid RF + VIF strategy was also evaluated by adding RF-ranked predictors only when the selected set satisfied the VIF threshold. All feature-selection operations were performed using the training data only, and the independent validation set was used only for model evaluation.
""".strip()

(OUTPUT_DIR / "Suggested_Methods_FeatureSelectionSensitivity.txt").write_text(methods_text, encoding="utf-8")

results_text = f"""
Suggested Results placeholder:

Feature-count sensitivity analysis showed whether increasing the selected predictor set beyond Top 25 produced meaningful gains in validation performance. The RF-ranking plus Pearson-correlation strategy was compared across Top 20, 25, 30, 35 and 40 predictors, and VIF-based alternatives were evaluated as supplementary robustness checks. The final model should be retained as Top 25 when larger feature sets provide only marginal gains in coffee-subclass macro F1 or overall accuracy, because the Top-25 model offers a better balance between accuracy, parsimony and interpretability.
""".strip()

(OUTPUT_DIR / "Suggested_Results_FeatureSelectionSensitivity.txt").write_text(results_text, encoding="utf-8")

print(caption_text)
print("\n---\n")
print("Recommendation:")
print(recommendation)

In [16]:

# ============================================================
# 10. Output manifest
# ============================================================
manifest = []
for p in sorted(OUTPUT_DIR.rglob("*")):
    if p.is_file():
        manifest.append({
            "file": str(p.relative_to(OUTPUT_DIR)),
            "size_kb": round(p.stat().st_size / 1024, 1),
        })

manifest_df = pd.DataFrame(manifest)
manifest_df.to_csv(OUTPUT_DIR / "OutputManifest_FeatureSelectionSensitivity.csv", index=False)
manifest_df


,file,size_kb
0,.gitkeep,0.0
1,audit_summary_metrics.csv,0.9
2,class_distribution_train_validation.csv,0.3
3,class_metrics_by_fold.csv,9.2
4,class_metrics_summary.csv,4.4
...,...,...
108,TableS12_Uncertainty_PerClass.xlsx,7.0
109,top30_missing_zero_risk_features.csv,2.1
110,validation_predictions_python_rf.csv,378.6
111,X_shap_sample.csv,191.1


In [ ]:
# =============================================================================
# OUTPUT MANIFEST
# =============================================================================
manifest_rows = []
for root in [TABLES_DIR, FIGURES_DIR, SUPPLEMENTARY_DIR]:
    if root.exists():
        for p in sorted(root.rglob("*")):
            if p.is_file():
                manifest_rows.append({
                    "folder": root.name,
                    "file": str(p.relative_to(root)),
                    "size_kb": round(p.stat().st_size / 1024, 1),
                })
manifest = pd.DataFrame(manifest_rows)
manifest_path = SUPPLEMENTARY_DIR / f"Manifest_{Path().resolve().name}.csv"
manifest.to_csv(manifest_path, index=False, encoding="utf-8-sig")
display(manifest.tail(30))
print("Manifest saved:", manifest_path)
